# BERT Sentiment Classification

## End-to-End NLP Use Case

Task:

```text
Given a customer review, predict sentiment:
positive, neutral, or negative.
```

Flow:

```text
review text -> tokenizer -> BERT -> sentiment prediction
```

## 1. What This Notebook Shows

This notebook demonstrates:

1. A small real-life review dataset
2. BERT tokenizer input format
3. `input_ids` and `attention_mask`
4. BERT sentiment prediction
5. Simple accuracy calculation

It uses a pretrained BERT sentiment model when `torch` and `transformers` are available.

## 2. Setup

If the required libraries are missing, install:

```python
!pip install transformers torch
```

Then restart the notebook kernel and run again.

In [1]:
import importlib.util


def print_table(headers, rows):
    """Print a compact table without pandas."""
    widths = [
        max(len(str(x)) for x in [header] + [row[i] for row in rows])
        for i, header in enumerate(headers)
    ]
    line = " | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for row in rows:
        print(" | ".join(str(x).ljust(widths[i]) for i, x in enumerate(row)))


# Check availability before importing heavy libraries.
# This keeps the notebook runnable even before setup is complete.
missing = [
    pkg for pkg in ["torch", "transformers"]
    if importlib.util.find_spec(pkg) is None
]

# READY controls whether later BERT cells should run or skip safely.
READY = len(missing) == 0

if READY:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    print("Ready: torch and transformers are available.")
else:
    print("Missing:", missing)
    print("Install with: pip install transformers torch")

Ready: torch and transformers are available.


## 3. Example Data

We use a tiny review dataset.

Each row has:

```text
text = review sentence
expected = human sentiment label
```

In [2]:
# Each review has text plus an expected label for evaluation.
reviews = [
    {
        "text": "The delivery was late but the product quality was excellent.",
        "expected": "positive",
    },
    {
        "text": "The phone battery drains quickly and the screen freezes.",
        "expected": "negative",
    },
    {
        "text": "The packaging was okay and the product works as expected.",
        "expected": "neutral",
    },
    {
        "text": "Customer support solved my issue within minutes.",
        "expected": "positive",
    },
    {
        "text": "I received the wrong item and nobody responded.",
        "expected": "negative",
    },
]

print_table(
    ["No.", "Review", "Expected"],
    [[i + 1, row["text"], row["expected"]] for i, row in enumerate(reviews)],
)

No. | Review                                                       | Expected
-----------------------------------------------------------------------------
1   | The delivery was late but the product quality was excellent. | positive
2   | The phone battery drains quickly and the screen freezes.     | negative
3   | The packaging was okay and the product works as expected.    | neutral 
4   | Customer support solved my issue within minutes.             | positive
5   | I received the wrong item and nobody responded.              | negative


## 4. Load Pretrained BERT

Model used:

```text
nlptown/bert-base-multilingual-uncased-sentiment
```

It predicts 1-star to 5-star sentiment.

We convert stars to labels:

```text
1-2 stars -> negative
3 stars   -> neutral
4-5 stars -> positive
```

In [3]:
# Pretrained BERT sentiment model from Hugging Face.
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
# MODEL_READY becomes True only after tokenizer and model load correctly.
MODEL_READY = False

if READY:
    try:
        # Tokenizer converts review text into BERT inputs.
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

        # Model converts BERT inputs into sentiment scores.
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
        model.eval()

        MODEL_READY = True
        print("Loaded:", MODEL_NAME)
    except Exception as error:
        print("Could not load model.")
        print("Reason:", error)
else:
    print("Skipping model load until libraries are installed.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded: nlptown/bert-base-multilingual-uncased-sentiment


## 5. Tokenizer Output

BERT tokenizer creates:

| Output | Meaning |
|---|---|
| `tokens` | readable token/subword pieces |
| `input_ids` | numeric IDs from BERT vocabulary |
| `attention_mask` | `1` for real tokens, `0` for padding |

BERT usually sees:

```text
[CLS] review tokens [SEP]
```

In [4]:
# Use the first review to inspect tokenizer output.
example = reviews[0]["text"]

if MODEL_READY:
    encoded = tokenizer(
        example,
        padding="max_length",
        truncation=True,
        max_length=24,
        return_tensors="pt",
    )

    # input_ids are the numeric vocabulary IDs that BERT consumes.
    input_ids = encoded["input_ids"][0].tolist()

    # attention_mask marks real tokens with 1 and padding with 0.
    attention_mask = encoded["attention_mask"][0].tolist()

    # Convert IDs back to readable token pieces for explanation.
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    print_table(
        ["Token", "Input ID", "Mask"],
        [
            [token, token_id, mask]
            for token, token_id, mask in zip(tokens, input_ids, attention_mask)
        ],
    )
else:
    print("[CLS] the delivery was late but the product quality was excellent . [SEP]")
    print("Tokenizer output will appear after model setup.")

Token     | Input ID | Mask
---------------------------
[CLS]     | 101      | 1   
the       | 10103    | 1   
delivery  | 43855    | 1   
was       | 10140    | 1   
late      | 12635    | 1   
but       | 10502    | 1   
the       | 10103    | 1   
product   | 20058    | 1   
quality   | 19468    | 1   
was       | 10140    | 1   
excellent | 42700    | 1   
.         | 119      | 1   
[SEP]     | 102      | 1   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   
[PAD]     | 0        | 0   


## 6. Prediction Function

The prediction function does this:

```text
text -> tokenizer -> BERT logits -> softmax probabilities -> label
```

In [5]:
def star_to_sentiment(star_label):
    """Convert a model star label into positive/neutral/negative."""
    star = int(star_label.split()[0])
    if star <= 2:
        return "negative"
    if star == 3:
        return "neutral"
    return "positive"


def predict_sentiment(text):
    """Run tokenizer and BERT model, then return sentiment output."""
    # Convert raw text into tensors expected by the BERT model.
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        # logits are raw scores before probability conversion.
        logits = model(**inputs).logits[0]

        # softmax converts raw scores into probabilities.
        probabilities = torch.softmax(logits, dim=0)

    # Pick the label with the highest probability.
    best_id = int(probabilities.argmax())
    star_label = model.config.id2label[best_id]

    return {
        "stars": star_label,
        "sentiment": star_to_sentiment(star_label),
        "confidence": round(float(probabilities[best_id]), 3),
    }


if MODEL_READY:
    print(predict_sentiment("The product is useful and delivery was fast."))
else:
    print("Prediction function is ready after model setup.")

{'stars': '4 stars', 'sentiment': 'positive', 'confidence': 0.55}


## 7. Run Prediction On All Reviews

In [6]:
if MODEL_READY:
    results = []

    # Predict each review one by one.
    for row in reviews:
        pred = predict_sentiment(row["text"])
        results.append(
            [
                row["text"],
                row["expected"],
                pred["sentiment"],
                pred["stars"],
                pred["confidence"],
            ]
        )

    print_table(
        ["Review", "Expected", "Predicted", "Stars", "Confidence"],
        results,
    )
else:
    print("Run this cell after the model is available.")

Review                                                       | Expected | Predicted | Stars   | Confidence
----------------------------------------------------------------------------------------------------------
The delivery was late but the product quality was excellent. | positive | positive  | 4 stars | 0.458     
The phone battery drains quickly and the screen freezes.     | negative | negative  | 2 stars | 0.476     
The packaging was okay and the product works as expected.    | neutral  | neutral   | 3 stars | 0.674     
Customer support solved my issue within minutes.             | positive | positive  | 5 stars | 0.412     
I received the wrong item and nobody responded.              | negative | negative  | 1 star  | 0.879     


## 8. Simple Evaluation

Accuracy:

```text
correct predictions / total examples
```

For a small classroom dataset, this is enough to explain evaluation.

In [7]:
if MODEL_READY:
    # Count correct predictions for a simple accuracy score.
    correct = 0

    # Predict each review one by one.
    for row in reviews:
        pred = predict_sentiment(row["text"])
        if pred["sentiment"] == row["expected"]:
            correct += 1

    print(f"Accuracy: {correct}/{len(reviews)} = {correct / len(reviews):.2f}")
else:
    print("Evaluation runs after model predictions are available.")

Accuracy: 5/5 = 1.00


## 9. Try A New Review

In [8]:
# Change this sentence to test another review.
my_review = "The laptop is fast, but the charger stopped working after two days."

if MODEL_READY:
    print("Review:", my_review)
    print("Prediction:", predict_sentiment(my_review))
else:
    print("Custom review:")
    print(my_review)
    print("Load the model to get a prediction.")

Review: The laptop is fast, but the charger stopped working after two days.
Prediction: {'stars': '2 stars', 'sentiment': 'negative', 'confidence': 0.485}
